In [1]:
import pandas as pd
import numpy as np

# 读取原始CSV文件
file_path = r"E:\EPFL_Courses\Data_Visualization\lyrical_emotion_trends.csv"
df = pd.read_csv(file_path)

# 步骤1: 数据清洗 - 移除缺失的decade和publicationYear
print(f"原始数据行数: {len(df)}")
df = df.dropna(subset=['decade', 'publicationYear'])
print(f"清洗后数据行数: {len(df)}")

# 查看decade分布
print("\n各个decade的数据量:")
decade_counts = df['decade'].value_counts().sort_index()
print(decade_counts)

# 步骤2: 创建各类词汇的指标
# 将word count转为布尔值指标 (有无特定词汇)
df['has_love_words'] = df['love_word_count'] > 0
df['has_swear_words'] = df['swear_word_count'] > 0
df['has_money_words'] = df['money_word_count'] > 0
df['has_sex_words'] = df['sex_word_count'] > 0
df['has_neutral_words'] = df['neutral_word_count'] > 0

# 步骤3: 创建情感指标
# 假设valence_mean > 0.5为积极情感，< 0.3为消极情感
df['positive_sentiment'] = df['valence_mean'] > 0.5
df['negative_sentiment'] = df['valence_mean'] < 0.3

# 步骤4: 按照decade进行分组统计
decade_stats = {}

# 提取我们需要处理的decade
decades = sorted(df['decade'].unique())

for decade in decades:
    decade_df = df[df['decade'] == decade]
    total_songs = len(decade_df)
    
    # 计算各类指标的百分比
    love_words_pct = 100 * decade_df['has_love_words'].mean()
    swear_words_pct = 100 * decade_df['has_swear_words'].mean()
    money_words_pct = 100 * decade_df['has_money_words'].mean()
    sex_words_pct = 100 * decade_df['has_sex_words'].mean()
    neutral_pct = 100 * decade_df['has_neutral_words'].mean()
    
    positive_sentiment_pct = 100 * decade_df['positive_sentiment'].mean()
    negative_sentiment_pct = 100 * decade_df['negative_sentiment'].mean()
    
    decade_stats[decade] = {
        'decade': decade,
        'total_songs': total_songs,
        'love_words': love_words_pct,
        'swear_words': swear_words_pct,
        'money_words': money_words_pct,
        'sex_words': sex_words_pct,
        'neutral_percentage': neutral_pct,
        'positive_sentiment': positive_sentiment_pct,
        'negative_sentiment': negative_sentiment_pct
    }

# 步骤5: 转换为DataFrame并保存为新的CSV
result_df = pd.DataFrame.from_dict(decade_stats, orient='index')
print("\n处理后的结果:")
print(result_df)

# 为可视化重新组织数据
# 创建每类指标的单独DataFrame
categories = [
    'love_words', 'positive_sentiment', 'negative_sentiment', 
    'money_words', 'swear_words', 'sex_words', 'neutral_percentage'
]

# 创建一个用于可视化的长格式数据
viz_data = []

for decade in decades:
    for category in categories:
        viz_data.append({
            'decade': decade,
            'category': category,
            'value': decade_stats[decade][category]
        })

viz_df = pd.DataFrame(viz_data)
print("\n可视化数据格式:")
print(viz_df.head(10))

# 保存为新的CSV文件
output_path = r"E:\EPFL_Courses\Data_Visualization\lyrics_viz_data.csv"
viz_df.to_csv(output_path, index=False)
print(f"\n已保存可视化数据到: {output_path}")

# 也保存聚合后的原始格式数据
agg_output_path = r"E:\EPFL_Courses\Data_Visualization\lyrics_aggregated_data.csv"
result_df.to_csv(agg_output_path)
print(f"已保存聚合数据到: {agg_output_path}")

# 显示一些基本统计信息和数据洞察
print("\n======= 数据洞察 =======")
for category in categories:
    avg = result_df[category].mean()
    min_val = result_df[category].min()
    max_val = result_df[category].max()
    min_decade = result_df[result_df[category] == min_val].index[0]
    max_decade = result_df[result_df[category] == max_val].index[0]
    
    print(f"\n{category}:")
    print(f"  平均值: {avg:.2f}%")
    print(f"  最低值: {min_val:.2f}% ({min_decade})")
    print(f"  最高值: {max_val:.2f}% ({max_decade})")
    print(f"  变化趋势: {'增加' if max_decade > min_decade else '减少' if max_decade < min_decade else '不确定'}")

原始数据行数: 1679972
清洗后数据行数: 1026277

各个decade的数据量:
decade
1900s       148
1910s        31
1920s         2
1930s         1
1940s        11
1950s       211
1960s       811
1970s      4138
1980s     19915
1990s    111491
2000s    573243
2010s    316173
2020s        14
2030s        63
2070s         1
2090s        14
2100s        10
Name: count, dtype: int64

处理后的结果:
      decade  total_songs  love_words  swear_words  money_words  sex_words  \
1900s  1900s          148    7.432432    30.405405    14.864865   6.081081   
1910s  1910s           31   25.806452     3.225806     3.225806   0.000000   
1920s  1920s            2    0.000000     0.000000     0.000000   0.000000   
1930s  1930s            1    0.000000     0.000000     0.000000   0.000000   
1940s  1940s           11    9.090909     9.090909     0.000000   0.000000   
1950s  1950s          211   21.327014     0.947867     7.582938   0.947867   
1960s  1960s          811    8.754624     2.712700     7.274969   1.479655   
1970s  1970s  